# Lab 1: Build the HR policy retrieval pipeline

## Business problem

Employees ask questions such as: **Can I expense a $300 train ticket without approval?** The answer exists in the HR policy, but the policy must be prepared for search before an LLM can use it.

## Mission

Build and inspect the retrieval pipeline that turns one HR document into searchable chunks.

By the end of this lab, you will be able to explain every moving piece:

1. Load the source document.
2. Preserve its policy sections.
3. Split only sections that are too large.
4. Attach metadata.
5. Create embeddings with one embedding model.
6. Store the chunks.
7. Retrieve evidence for a real HR question.

The AI engineering decision for this project is structure-aware recursive chunking. The LLMOps responsibility is to implement it reproducibly, test it, and detect when a change damages retrieval quality.


## Exercise 1: Load and inspect the source

**Mission:** Confirm what entered the pipeline before transforming it.

**Why it matters:** A retrieval system cannot repair missing or incorrectly parsed source content.


In [ ]:
from pathlib import Path

policy_path = Path("hr_policy.txt")
policy_text = policy_path.read_text(encoding="utf-8")

print("Characters:", len(policy_text))
print("Words:", len(policy_text.split()))
print()
print(policy_text[:500])


### Inspect the result

- Is the title present?
- Are the section headings present?
- Is the text readable?

If the input were a PDF or DOCX file, a parser would first extract this same text and metadata. Chunking begins after parsing succeeds.


## Exercise 2: Preserve the policy structure

**Mission:** Turn the handbook into complete policy sections before applying a size limit.

**Why it matters:** A heading such as `Section 3: Reimbursements` gives meaning to the paragraphs below it. Splitting only by character count can separate the heading from its policy.


In [ ]:
lines = policy_text.splitlines()

document_title = ""
intro_lines = []
sections = []
current_heading = ""
current_lines = []

for line in lines:
    if line.startswith("# "):
        document_title = line[2:]
    elif line.startswith("## "):
        if current_heading:
            sections.append({"heading": current_heading, "text": "\n".join(current_lines).strip()})
        current_heading = line[3:]
        current_lines = []
    elif current_heading:
        current_lines.append(line)
    elif line.strip():
        intro_lines.append(line)

if current_heading:
    sections.append({"heading": current_heading, "text": "\n".join(current_lines).strip()})

print("Document:", document_title)
print("Policy sections:", len(sections))
print()
for section in sections:
    print(section["heading"], "|", len(section["text"]), "characters")


### What this proves

We now have named policy sections, not anonymous pieces of text. This structure will become searchable metadata and later support citations.


## Exercise 3: Split only oversized sections

**Mission:** Apply a controlled size limit without discarding the section boundaries.

**Experiment:** Change `CHUNK_SIZE` and `CHUNK_OVERLAP`, rerun the cell, and inspect what changes.

This library measures characters. The lecture discusses tokens because model limits and costs are token based. Character limits are used here so the splitting behavior remains visible to Python beginners.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []

for section in sections:
    text_to_split = section["heading"] + "\n" + section["text"]
    section_chunks = splitter.split_text(text_to_split)

    for position, chunk_text in enumerate(section_chunks):
        chunks.append(
            {
                "text": chunk_text,
                "source": policy_path.name,
                "section": section["heading"],
                "position": position,
            }
        )

print("Sections became", len(chunks), "chunks")
print()
for chunk in chunks:
    print(chunk["section"], "| position", chunk["position"], "|", len(chunk["text"]), "characters")
    print(chunk["text"])
    print()


### Inspect the result

- Did each short policy section remain complete?
- If a section was split, did both chunks retain the section metadata?
- Does increasing the overlap duplicate more text?

Overlap helps preserve context only when a split occurs. Too much overlap creates duplicate evidence, increases embedding cost, and can crowd retrieval results.


## Exercise 4: Create one embedding and inspect it

**Mission:** Convert one chunk into the numeric representation used for similarity search.

**Why it matters:** Document chunks and user questions must use the same embedding model and configuration so their vectors have the same dimensions and live in the same vector space.


In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()
EMBEDDING_MODEL = "text-embedding-3-small"

response = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=chunks[0]["text"],
)

first_vector = response.data[0].embedding

print("Vector dimensions:", len(first_vector))
print("First 10 values:", first_vector[:10])


### What this proves

The vector is a list of numbers. Its dimensions are not physical directions that humans can draw. Together, the values position the meaning of the text in the embedding model's vector space.


## Exercise 5: Embed and index every chunk

**Mission:** Store each chunk with its vector and metadata.

**Why it matters:** Production systems must be able to trace a retrieved vector back to the source text and policy section.


In [ ]:
import numpy as np

chunk_texts = []
for chunk in chunks:
    chunk_texts.append(chunk["text"])

response = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=chunk_texts,
)

chunk_vectors = []
for item in response.data:
    chunk_vectors.append(item.embedding)

print("Indexed chunks:", len(chunk_vectors))
print("Dimensions per chunk:", len(chunk_vectors[0]))


## Exercise 6: Retrieve the evidence

**Mission:** Find the policy chunks most similar to an employee question.

**Experiment:** Change `TOP_K` to 1, 3, and 5. Notice how retrieving more evidence can add useful context but can also add unrelated text.


In [ ]:
QUESTION = "Can I expense a $300 train ticket without approval?"
TOP_K = 3

question_response = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=QUESTION,
)
question_vector = question_response.data[0].embedding

def cosine_similarity(vector_a, vector_b):
    a = np.array(vector_a)
    b = np.array(vector_b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

results = []

for index in range(len(chunks)):
    score = cosine_similarity(question_vector, chunk_vectors[index])
    results.append({"score": score, "chunk": chunks[index]})

results.sort(key=lambda result: result["score"], reverse=True)
top_results = results[:TOP_K]

for result in top_results:
    print("Score:", round(result["score"], 3))
    print("Source:", result["chunk"]["source"])
    print("Section:", result["chunk"]["section"])
    print(result["chunk"]["text"])
    print()


## Lab 1 checkpoint

The pipeline can now answer the retrieval question: **Which policy text should an LLM receive?**

Before moving on, verify that `Section 3: Reimbursements` appears first. If it does not, inspect the source, chunks, metadata, embedding model, question, and retrieval settings. That investigation is part of operating a RAG system.
